# Modèle de langage et génération de séquence

Ce TP a pour but de vous familiariser avec le concept de modèle de langage et de génération de séquence.

À partir d'un corpus de textes écrits par Voltaire, nous allons apprendre un modèle récurrent basé sur les séquences de caractères.

## Vérification de l'utilisation de GPU

Allez dans le menu `Exécution > Modifier le type d'execution` et vérifiez que l'on est bien en Python 3 et que l'accélérateur matériel est configuré sur « GPU ».

In [ ]:
!nvidia-smi

## Récupération des données

Les différents texte de Voltaire qui constituent le corpus ont été obtenus à partir de gutenberg.org. Les headers, footers ainsi que les préfaces ont été préalablement enlevés : on ne voudrait pas que notre modèle de langage apprenne à écrire les disclaimers de gutenberg.org ou les préface de l'éditeur.

On utilisera uniquement le fichier `dataset-voltaire/voltaire_clean.txt`.

In [ ]:
!rm -rf dataset-voltaire
!git clone https://github.com/nzmonzmp/dataset-voltaire.git
print("─" * 50)
!ls -l dataset-voltaire/
print("─" * 50)
!cat dataset-voltaire/voltaire_clean.txt


In [ ]:
# Utilisez cette cellule pour explorer un peu les données.
!wc dataset-voltaire/voltaire_clean.txt

In [ ]:
# Décommentez pour télécharger le fichier
# import google.colab
# google.colab.files.download('dataset-voltaire/concat_voltaire.txt')

## Installation et import de PyTorch Lightning et des autres librairies nécessaires

In [ ]:
!pip install -q lightning torchmetrics

In [ ]:
import pathlib

import lightning
import numpy
import torch
import torchmetrics
from lightning.pytorch.callbacks import Callback
from lightning.pytorch.loggers import CSVLogger
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

## Chargement des données et extraction du vocabulaire

- *Chargez le texte du ficher en miniscules dans la variable `text`*
- *Extrayez tous les caractères utilisés dans `text` dans la liste `index_to_char`, triés par ordre alphabétique. Cette liste sert pour passer de la représentation numérique d'un caractère au caractère lui-même*
- *Construisez un dictionnaire `char_to_index` qui va permettre de passer d'un caractère à sa représentation numérique*

In [ ]:
text = "toto"
index_to_char = ["o", "t"]
char_to_index = {"o": 0, "t": 1}

### Solution

In [ ]:
text = pathlib.Path("dataset-voltaire/voltaire_clean.txt").read_text().lower()
print(f"{text[:100]}…")
print(f"Nombre de caractères : {len(text)}")

In [ ]:
index_to_char = sorted(set(text))
print(index_to_char)
print(f"Taille du dictionnaire : {len(index_to_char)}")

In [ ]:
char_to_index = {c: i for i, c in enumerate(index_to_char)}
print(char_to_index)

## Prétraitements

Découpage du texte en séquences de 30 caractères, tous les 3 caractères. Les cibles seront les 31èmes caractères

In [ ]:
maxlen = 30
step = 3
sentences = []
next_chars = []
for i in range(0, len(text) - maxlen, step):
  sentences.append(text[i: i + maxlen])
  next_chars.append(text[i + maxlen])
print(f"Nombre de séquences : {len(sentences)}")

In [ ]:
print(sentences[0])

## Création des tableaux d'entrée et de sortie

*Créez les variables `X` et `y` qui contiennent les données d'entrée et de sortie encodées.*

In [ ]:
# Votre code ici

### Solution

In [ ]:
def encode(sentence: str) -> torch.Tensor:
  return torch.tensor([char_to_index[char] for char in sentence])


X = torch.stack([encode(sentence) for sentence in sentences])
y = encode(next_chars)

print(X.shape, y.shape)

## Préparation du modèle

*Essayez différents modèles ([`LSTM`](https://docs.pytorch.org/docs/stable/generated/torch.nn.LSTM.html), [`GRU`](https://docs.pytorch.org/docs/stable/generated/torch.nn.GRU.html), [`RNN`](https://docs.pytorch.org/docs/stable/generated/torch.nn.RNN.html)) de différentes tailles.*

*Laissez quelques itérations à l'algorithme avant d'essayer une nouvelle configuration, ou utilisez [Optuna](https://optuna.org/) pour le faire automatiquement.*

L'objectif étant évidemment de minimiser la loss : Plus elle est basse, plus le modèle est proche du modèle de langage de Voltaire.


In [ ]:
class LanguageModel(lightning.LightningModule):
  """Lightning wrapper: predict the character following a sequence."""

  def __init__(self,
               model: nn.Module,
               learning_rate: float = 1e-3,
               sequence_len: int = maxlen,
               num_classes: int = len(index_to_char)) -> None:
    super().__init__()
    self.save_hyperparameters(ignore=["model"])
    self.model = model
    # Une entrée d'exemple permet à Lightning d'afficher la forme des tenseurs
    # d'entrée et de sortie de chaque couche dans le résumé du modèle
    self.example_input_array = torch.zeros(1, sequence_len, dtype=torch.long)
    self.accuracy = torchmetrics.Accuracy(task="multiclass",
                                          num_classes=num_classes)

  def forward(self, sentences: torch.Tensor) -> torch.Tensor:
    return self.model(sentences)

  def training_step(self, batch: tuple[torch.Tensor, torch.Tensor],
                    batch_index: int) -> torch.Tensor:
    sentences, next_chars = batch
    logits = self(sentences)
    loss = nn.functional.cross_entropy(logits, next_chars)
    self.accuracy(logits, next_chars)
    self.log("train_loss", loss, on_step=False, on_epoch=True, prog_bar=True)
    self.log("train_accuracy", self.accuracy, on_step=False, on_epoch=True,
             prog_bar=True)
    return loss

  def configure_optimizers(self) -> torch.optim.Optimizer:
    return torch.optim.Adam(self.parameters(),
                            lr=self.hparams.learning_rate)

In [ ]:
class CharRNN(nn.Module):
  """Embed the characters, run a recurrent network, score the next one."""

  def __init__(self,
               embedding_dim: int = 8,
               lstm_hidden_dim: int = 128,
               num_layers: int = 1) -> None:
    super().__init__()
    self.embedding = nn.Embedding(len(index_to_char), embedding_dim)
    self.lstm = nn.LSTM(embedding_dim,
                        lstm_hidden_dim,
                        num_layers=num_layers,
                        batch_first=True)
    self.head = nn.Linear(lstm_hidden_dim, len(index_to_char))

  def forward(self, sentences: torch.Tensor) -> torch.Tensor:
    embedded = self.embedding(sentences)
    outputs, _ = self.lstm(embedded)
    # Seule la sortie du dernier pas de temps nous intéresse : c'est le résumé
    # de toute la séquence
    return self.head(outputs[:, -1])


def build_model(embedding_dim: int = 8,
                lstm_hidden_dim: int = 128,
                learning_rate: float = 1e-3
                ) -> lightning.LightningModule:
  return LanguageModel(CharRNN(embedding_dim, lstm_hidden_dim),
                       learning_rate=learning_rate)

## Génération de texte

Le callback qui suit s'exécute toutes les `every` epochs, un paramètre que l'on peut donner pendant son initialisation.

Il génère des caractères un à un, en décalant progressivement l'entrée donnée pour qu'elle fasse toujours `maxlen` caractères.

[`torch.multinomial`](https://docs.pytorch.org/docs/stable/generated/torch.multinomial.html) permet d'échantillonner depuis une distribution de probabilité, c'est donc notre outil principal pour exploiter la sortie du réseau : le softmax appliqué aux logits définit en effet une telle distribution.

In [ ]:
class SamplingCallback(Callback):
  def __init__(self, n_steps: int = 200, every: int = 10) -> None:
    super().__init__()
    self.n_steps = n_steps
    self.every = every

  @torch.no_grad()
  def on_train_epoch_end(self,
                         trainer: lightning.Trainer,
                         pl_module: lightning.LightningModule) -> None:
    epoch = trainer.current_epoch + 1
    if epoch % self.every == 0:
      print()
      print("─" * 50)
      print(f"Après {epoch} epochs :")
      start_index = numpy.random.randint(0, len(text) - maxlen - 1)
      sentence = text[start_index:start_index + maxlen]
      print("─" * 50)
      print(f"Génération à partir de : « {sentence} »")
      print("─" * 50)
      print(sentence, end="")

      pl_module.eval()
      for _ in range(self.n_steps):
        logits = pl_module(encode(sentence)[None, :].to(pl_module.device))
        word_probas = logits.softmax(dim=-1)
        next_index = int(torch.multinomial(word_probas, 1))
        next_char = index_to_char[next_index]
        sentence = sentence[1:] + next_char
        print(next_char, end="")
      pl_module.train()
      print()

## Apprentissage

In [ ]:
batch_size = 4096
train_loader = DataLoader(TensorDataset(X, y),
                          batch_size=batch_size,
                          shuffle=True)

model = build_model()
trainer = lightning.Trainer(max_epochs=100,
                            accelerator="auto",
                            devices=1,
                            logger=CSVLogger("logs", name="char_rnn"),
                            enable_checkpointing=False,
                            callbacks=[SamplingCallback(n_steps=200,
                                                        every=10)])
trainer.fit(model, train_loader)

## Une solution avec plusieurs couches de LSTM

In [ ]:
def build_deep_model(embedding_dim: int = 8,
                     lstm_hidden_dim: int = 128,
                     learning_rate: float = 1e-3
                     ) -> lightning.LightningModule:
  return LanguageModel(CharRNN(embedding_dim, lstm_hidden_dim, num_layers=2),
                       learning_rate=learning_rate)

In [ ]:
deep_model = build_deep_model()
deep_trainer = lightning.Trainer(
    max_epochs=100,
    accelerator="auto",
    devices=1,
    logger=CSVLogger("logs", name="deep_char_rnn"),
    enable_checkpointing=False,
    callbacks=[SamplingCallback(n_steps=200, every=10)])
deep_trainer.fit(deep_model, train_loader)